# Evaluation: parsing (VLM only vs Docling anchor)

## Setup

In [ ]:
import json, re, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

ROOT   = Path.cwd().parent
PARSED = ROOT / "data/parsed_NO_EDIT/machine_learning"
GOLDEN = ROOT / "data/eval/test_jsonfiles/golden"
ANCHOR = ROOT / "data/eval/test_jsonfiles/anchor"
EVAL_OUT = ROOT / "data/eval"
EVAL_OUT.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    {"vorlesung": "SVM", "methode": "ohne Docling",
     "parse":  PARSED / "ML_5_svm/ML_5_svm_chunks.json",
     "golden": GOLDEN / "ML_5_svm_golden_parse.json"},
    {"vorlesung": "SVM", "methode": "mit Docling",
     "parse":  ANCHOR / "ML_5_svm_chunks_docling_anchor.json",
     "golden": GOLDEN / "ML_5_svm_golden_parse.json"},
    {"vorlesung": "Neuronale Netze", "methode": "ohne Docling",
     "parse":  PARSED / "ML_9_neuronale_netze/ML_9_neuronale_netze_chunks.json",
     "golden": GOLDEN / "ML_9_neuronale_netze_golden_parse.json"},
    {"vorlesung": "Neuronale Netze", "methode": "mit Docling",
     "parse":  ANCHOR / "ML_9_neuronale_netze_chunks_docling_anchor.json",
     "golden": GOLDEN / "ML_9_neuronale_netze_golden_parse.json"},
]

def load_pairs(cfg):
    parse  = json.loads(Path(cfg["parse"]).read_text(encoding="utf-8"))
    golden = json.loads(Path(cfg["golden"]).read_text(encoding="utf-8"))
    by_id  = {c["id"]: c for c in parse}
    pairs  = [(g, by_id[g["slide_id"]]) for g in golden if g["slide_id"] in by_id]
    return parse, golden, by_id, pairs

print("Configurations:")
for cfg in CONFIGS:
    parse, golden, by_id, pairs = load_pairs(cfg)
    print(f"{cfg['vorlesung']} | {cfg['methode']} | " 
          f"parse={len(parse)}  golden={len(golden):3d}  pairs={len(pairs)}")

## Normalise

In [ ]:

LATEX = {
    r"\alpha": "α", r"\beta": "β", r"\gamma": "γ", r"\delta": "δ",
    r"\epsilon": "ε", r"\varepsilon": "ε", r"\zeta": "ζ", r"\eta": "η",
    r"\theta": "θ", r"\kappa": "κ", r"\lambda": "λ", r"\mu": "μ",
    r"\nu": "ν", r"\xi": "ξ", r"\pi": "π", r"\rho": "ρ", r"\sigma": "σ",
    r"\tau": "τ", r"\phi": "φ", r"\chi": "χ", r"\psi": "ψ", r"\omega": "ω",
    r"\leq": "≤", r"\le": "≤", r"\geq": "≥", r"\ge": "≥",
    r"\neq": "≠", r"\ne": "≠", r"\approx": "≈", r"\times": "×",
    r"\pm": "±", r"\infty": "∞", r"\sum": "∑", r"\partial": "∂",
    r"\nabla": "∇", r"\in": "∈", r"\rightarrow": "→", r"\to": "→",
}

def normalize(t: str) -> str:
    t = unicodedata.normalize("NFC", t)
    t = t.lower()
    for cmd in sorted(LATEX, key=len, reverse=True):    
        t = re.sub(re.escape(cmd) + r"(?![a-z])", LATEX[cmd], t)
    t = t.replace(",,", "").replace("``", "").replace("''", "")  
    t = re.sub(r'[„“”‚‘’»«"]', "", t)                    
    t = re.sub(r"\$+", " ", t)                           
    t = re.sub(r"[*#`>~]", " ", t)                   
    t = re.sub(r"…", "...", t)                          
    t = re.sub(r"[‐-―−]", "-", t)         
    t = re.sub(r"[•·‣▪]", " ", t)                       
    t = re.sub(r"(?m)^\s*[-→]\s+", " ", t)               
    t = re.sub(r";", " ", t)                            
    t = re.sub(r"\s*([.,:!?=≠≥≤≈×±])\s*", r"\1", t)     
    t = re.sub(r"\s+", " ", t)                        
    return t.strip()

## Extract text and blocks from chunks

Defining helpers to split a chunk into plain text nuggets or [GRAFIK]/[FORMEL]/[CODE] blocks, isolating the text content for string based recall

In [ ]:
BLOCK_PREFIXES = ("[GRAFIK]", "[FORMEL]", "[CODE]")

def is_text_nugget(n: str) -> bool:
    return not n.lstrip().startswith(BLOCK_PREFIXES)

def build_parsetext(chunk: dict) -> str:
    parts = []
    if chunk.get("title"):
        parts.append("Titel: " + chunk["title"])
    parts.append(chunk.get("page_content", "").replace("\\n", "\n"))
    return "\n".join(parts)


## Metric: recall

How many of a slide's gold text facts show up (as a normalised substring) in the parsed chunk

In [ ]:
def recall_counts(text_nuggets, chunk):
    if not text_nuggets:
        return 0, 0
    chunk = normalize(chunk)
    hits = sum(normalize(nugget) in chunk for nugget in text_nuggets)
    return hits, len(text_nuggets)

## Recall over all texts

Computing pooled text recall per config, comparing VLM only against Docling anchored parsing on plain text

In [ ]:
text_rows = []

for cfg in CONFIGS:
    _, _, _, pairs = load_pairs(cfg)
    total_hits = 0
    total_nuggets = 0

    for gold_slide, parsed_chunk in pairs:
        text_nuggets = [n for n in gold_slide["text"] if is_text_nugget(n)]
        parse_text = build_parsetext(parsed_chunk)
        hits, n = recall_counts(text_nuggets, parse_text)
        total_hits += hits
        total_nuggets += n

    recall = total_hits / total_nuggets
    text_rows.append({
        "Vorlesung": cfg["vorlesung"], 
        "Methode": cfg["methode"],
        "Text-Recall": recall, 
        "Text-Fakten": total_nuggets,
    })

text_df = pd.DataFrame(text_rows)
text_df

## LLM as a judge: setup

Configuring the LLM as judge client, used for the block modalities where exact string matching fails (formulas, code)

In [ ]:
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(os.path.join(os.getcwd(), "..", ".env"), override=True)

GATEWAY_URL = os.getenv("GATEWAY_URL", "")
BEARER_TOKEN = os.getenv("BEARER_TOKEN", "")
JUDGE_MODEL = os.getenv("INFERENCE_MODEL_GATEWAY", "") 

client = OpenAI(base_url=GATEWAY_URL, api_key=BEARER_TOKEN)

print("Judge-LLM Model:", JUDGE_MODEL)

An LLM call (with retries) that decides semantically whether a gold element is covered by the parsed slide (covered or missing)

In [ ]:
BLOCK_TYPES = ["grafik", "formel", "code"]

def build_fulltext(chunk):
    titel = "Titel: " + chunk["title"] + "\n" if chunk.get("title") else ""
    return titel + chunk.get("page_content", "")

cache = {}

def judge_item(element, parse_text, retries=2):
    key = (element, parse_text)
    if key in cache:
        return cache[key]

    prompt = f"""
Du bist ein Evaluator für Informations-Recall in Folien-Parsing.

Aufgabe:
Prüfe ob, die Information des gegebenen Elements irgendwo im geparsten Text enthalten ist.

Wichtig:
- Es spielt KEINE Rolle, wo die Information steht (Text, Grafikbeschreibung, Formel, Code).
- Es spielt KEINE Rolle, wie sie formuliert ist.
- Entscheidend ist nur semantische Ähnlichkeit.
- Zusaetzlicher Inhalt im Parse ist irrelevant.
- Du bewertest nur Recall: enthalten oder nicht enthalten.

Element:
{element}

Geparste Folie:
{parse_text}

Antworte strikt als JSON:
{{"verdict": "covered" oder "missing", "reason": "kurze Begründung"}}
"""

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0,
            )
            parsed = json.loads(resp.choices[0].message.content)
            verdict = str(parsed.get("verdict", "")).strip().lower()
            if verdict not in ("covered", "missing"):
                raise ValueError(f"unerwartetes verdict: {verdict!r}")
            result = {"verdict": verdict, "reason": parsed.get("reason", "")}
            cache[key] = result
            return result
        except Exception as e:
            print(f"    Try {attempt + 1} failed: {e}")

    return {"verdict": "invalid", "reason": f"Judge invalid after {retries} tries"}


In [ ]:
_, golden, by_id, _ = load_pairs(CONFIGS[0])

# Pick the first block element (grafik/formel/code) we can find anywhere in the golden set and use it as a test
test = None
for slide in golden:
    for block_type in BLOCK_TYPES:
        elements = slide.get(block_type, [])
        if elements:
            test = elements[0]
            break
    if test is not None:
        break

contains_text  = "Titel: Testfolie\n" + test 
unrelated_text = "Titel: Organisatorisches\nDie Klausur findet am 15. Maerz statt; bitte rechtzeitig anmelden."

print("ELEMENT:\n", test, "\n")
print("contains it ->", judge_item(test, contains_text))
print("unrelated ->", judge_item(test, unrelated_text))

## Block recall: formula and code

Running the judge to measure formula and code recall per config, the modalities that need semantic matching rather than string matching

In [ ]:
RUN_JUDGE = True
JUDGE_TYPES = ["formel", "code"]

block_rows = []   

if RUN_JUDGE:
    for cfg in CONFIGS:
        _, golden, by_id, _ = load_pairs(cfg)

        for modality in JUDGE_TYPES:
            covered = 0
            total = 0
            for g in golden:
                sid = g["slide_id"]
                if sid not in by_id:
                    continue
                parse_text = build_fulltext(by_id[sid])
                for element in g.get(modality, []):
                    verdict = judge_item(element, parse_text)["verdict"]
                    if verdict in ("covered", "missing"):
                        total += 1
                        if verdict == "covered":
                            covered += 1

            recall = covered / total if total else None
            block_rows.append({
                "Vorlesung": cfg["vorlesung"],
                "Methode": cfg["methode"],
                "Modalität": modality,
                "Recall": recall,
                "Treffer": covered,
                "Total": total,
            })

block_df = pd.DataFrame(block_rows)
block_df

## Overall table: recall with and without Docling

Pooling text/formula/code recall into one table with and without Docling, including the deltas

In [ ]:
METHODS = ["ohne Docling", "mit Docling"]
MODALITIES = ["Text", "Formel", "Code"]

def text_pooled(method):
    covered = 0
    total = 0
    for cfg in CONFIGS:
        if cfg["methode"] != method:
            continue
        _, _, _, pairs = load_pairs(cfg)
        for g, p in pairs:
            text_nuggets = [n for n in g["text"] if is_text_nugget(n)]
            hits, n = recall_counts(text_nuggets, build_parsetext(p))
            covered += hits
            total += n
    return covered, total

def block_pooled(method, modality_label):
    sub = block_df[(block_df["Methode"] == method) & (block_df["Modalität"] == modality_label)]
    return int(sub["Treffer"].sum()), int(sub["Total"].sum())

counts = {}
for method in METHODS:
    text_covered, text_total = text_pooled(method)
    formel_covered, formel_total = block_pooled(method, "formel")
    code_covered, code_total = block_pooled(method, "code")
    counts[method] = {
        "Text":   {"covered": text_covered,   "total": text_total},
        "Formel": {"covered": formel_covered, "total": formel_total},
        "Code":   {"covered": code_covered,   "total": code_total},
    }

rows = []
for modality in MODALITIES:
    without = counts["ohne Docling"][modality]
    with_docling = counts["mit Docling"][modality]
    recall_without = without["covered"] / without["total"]
    recall_with = with_docling["covered"] / with_docling["total"]
    rows.append({
        "Modalität": modality,
        "n": without["total"],
        "ohne Docling": recall_without,
        "mit Docling": recall_with,
        "Δ (mit - ohne)": recall_with - recall_without,
    })

without_covered = sum(counts["ohne Docling"][m]["covered"] for m in MODALITIES)
without_total = sum(counts["ohne Docling"][m]["total"] for m in MODALITIES)
with_covered = sum(counts["mit Docling"][m]["covered"] for m in MODALITIES)
with_total = sum(counts["mit Docling"][m]["total"] for m in MODALITIES)
recall_without = without_covered / without_total
recall_with = with_covered / with_total
rows.append({
    "Modalität": "Gesamt",
    "n": without_total,
    "ohne Docling": recall_without,
    "mit Docling": recall_with,
    "Δ (mit - ohne)": recall_with - recall_without,
})

cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(EVAL_OUT / "parsing_docling_vergleich.csv", index=False, encoding="utf-8")
print("gespeichert:", EVAL_OUT / "parsing_docling_vergleich.csv")

cmp_df.style.format({"n": "{:.0f}", "ohne Docling": "{:.3f}",
                     "mit Docling": "{:.3f}", "Δ (mit - ohne)": "{:+.3f}"}).hide(axis="index") 